# Accuracy of Our Models on Our Survey Answers

### Choose Models to Test

In [17]:
from Testing.shap.model_wrapper import Model

models = [
    Model('political', 'regression', train='Baseline'),
]

Initialized model political (DistilBERT regression Baseline)


### Testing Function for Each Trait

In [18]:
from evaluate import load
import numpy as np
import pandas as pd

accuracy = load("accuracy")

batch_size = 64

for model in models:
    trait = model.name.split(" ")[0]
    df = pd.read_parquet(f"Training/DATA/{trait}/{'short_' if trait == 'political' else ''}test.parquet")
    if trait == 'political':
        trait = 'political_view'
    if trait == 'mbti':
        trait = 'type'
    invert_labels = {v: k for k, v in model.label_map.items()} if model.label_map != None else lambda x: x

    all_preds = []
    all_refs = df[trait].map(invert_labels).to_numpy()

    for start in range(0, len(df), batch_size):
        batch = df['text'].iloc[start:start + batch_size].tolist()
        outputs = model.predict(batch)

        match model.type:
            case 'classification' | 'convolution':
                outputs = np.argmax(outputs, axis=1)
            case 'regression' | 'classreg':
                outputs = np.round(outputs)

        all_preds.extend(outputs)

    acc = accuracy.compute(predictions=all_preds, references=all_refs)

    print(f"{model.name} - accuracy: {acc}")


political (DistilBERT regression Baseline) - accuracy: {'accuracy': 0.22787071625791783}
